# 04.04 PaddleOCR PP-OCRv6 Fine-Tuning & Text Extraction

**Phase 04 — Document Processing & OCR**  
**Authoritative Engine:** PaddleOCR PP-OCRv6 (`PP-OCRv6_medium_rec` SVTR Recognition + `PP-OCRv6_medium_det` DBNet Detection — Inference-Only).  
**Hardware Target:** NVIDIA RTX 3050 Laptop GPU (4 GB VRAM), Python 3.12, PaddlePaddle GPU 3.3.1, PaddleOCR 3.7.0, PaddleX 3.7.2.

### Objectives & Strict Governance
1. Audit `PP-OCRv6_medium_det` (DBNet, **Pretrained / Inference-Only**) separately from `PP-OCRv6_medium_rec` (LCNetV4 + EncoderWithLightSVTR, **Fine-Tuning Target**).
2. Fine-tune `PP-OCRv6_medium_rec` on OFPHIL manifests (`train_manifest_v2.csv`: 15,043 crops; `val_manifest_v2.csv`: 3,536 crops) with `CTCLoss`, `AdamW`, `CosineAnnealingDecay`, and resumable checkpointing under `models/phase_04/paddleocr_recognition/`.
3. Enforce strict document-level partition isolation — `test_manifest_v2.csv` (2,778 crops) is never touched until final held-out evaluation.
4. Run actual OCR text, bounding-box, and confidence extraction and export structured entities (`data/interim/document_processing/extracted_ocr/ocr_extracted_entities.json`) for Notebooks 05 and 06.


### 1. Environment, Windows DLL Setup & Centralized Paths


In [1]:
import os
import sys
import gc
import re
import ast
import time
import math
import json
import yaml
import random
import shutil
import warnings
import unicodedata
from pathlib import Path

# Windows DLL setup: if nvidia/cudnn/bin is installed (Paddle 3.3+ cu126), let Paddle load its own cuDNN 9.5;
# otherwise fall back to torch/lib for cuDNN 9 DLL resolution.
_nvidia_cudnn_bin = Path(sys.prefix) / 'Lib' / 'site-packages' / 'nvidia' / 'cudnn' / 'bin'
_torch_lib = Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib'
_DLL_HANDLES = []
if not _nvidia_cudnn_bin.exists() and _torch_lib.exists():
    if hasattr(os, 'add_dll_directory'):
        try:
            _DLL_HANDLES.append(os.add_dll_directory(str(_torch_lib)))
        except OSError:
            pass
    os.environ['PATH'] = str(_torch_lib) + os.pathsep + os.environ.get('PATH', '')
    try:
        import torch as _torch_dll_preload  # noqa: F401
    except Exception:
        pass

warnings.filterwarnings('ignore')
os.environ.setdefault('GLOG_minloglevel', '3')
os.environ.setdefault('PADDLE_PDX_LOGGING_LEVEL', 'ERROR')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import paddle
import paddle.nn.functional as F

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
paddle.seed(RANDOM_SEED)

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

# Centralized Phase 04 PaddleOCR model artifact directory
MODEL_DIR = paths.paddleocr_rec_model_dir
BEST_CKPT_DIR = MODEL_DIR / 'best_accuracy'
LAST_CKPT_DIR = MODEL_DIR / 'last_checkpoint'
HISTORY_CSV_PATH = MODEL_DIR / 'paddleocr_training_history.csv'
CONFIG_YAML_PATH = MODEL_DIR / 'config.yaml'
TRAINING_STATE_JSON = MODEL_DIR / 'training_state.json'

EXTRACTED_OCR_DIR = paths.interim_doc_proc / 'extracted_ocr'
OUT_ENTITIES_JSON = EXTRACTED_OCR_DIR / 'ocr_extracted_entities.json'
BENCHMARK_OUT_DIR = paths.model_benchmarks_dir / 'paddleocr_finetuned'
FINAL_REPORT_PATH = BENCHMARK_OUT_DIR / 'paddleocr_finetuned_benchmark_report.json'
TEST_PREDICTIONS_CSV = BENCHMARK_OUT_DIR / 'paddleocr_finetuned_test_predictions.csv'

for _d in (MODEL_DIR, BEST_CKPT_DIR, LAST_CKPT_DIR, EXTRACTED_OCR_DIR, BENCHMARK_OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST = paths.splits_dir / 'train_manifest_v2.csv'
VAL_MANIFEST = paths.splits_dir / 'val_manifest_v2.csv'
TEST_MANIFEST = paths.splits_dir / 'test_manifest_v2.csv'

PADDLEX_MODELS_ROOT = Path.home() / '.paddlex' / 'official_models'
PRETRAINED_DET_DIR = PADDLEX_MODELS_ROOT / 'PP-OCRv6_medium_det'
PRETRAINED_REC_DIR = PADDLEX_MODELS_ROOT / 'PP-OCRv6_medium_rec'

HAS_CUDA = bool(paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0)
ACTIVE_DEVICE = 'gpu:0' if HAS_CUDA else 'cpu'
paddle.set_device(ACTIVE_DEVICE)

def get_gpu_vram_mb() -> float:
    if HAS_CUDA:
        try:
            return float(paddle.device.cuda.max_memory_allocated(0) / (1024 ** 2))
        except Exception:
            return 0.0
    return 0.0

PIPELINE_STATUS = 'PENDING MANUAL EXECUTION'
PIPELINE_ERROR = None

print(f"Project Root        : {paths.root}")
print(f"PaddlePaddle        : {paddle.__version__} | Device: {ACTIVE_DEVICE.upper()}")
print(f"Model Artifacts Dir : {MODEL_DIR}")
print(f"Manifests Present   : Train={TRAIN_MANIFEST.exists()} | Val={VAL_MANIFEST.exists()} | Test={TEST_MANIFEST.exists()}")


Project Root        : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
PaddlePaddle        : 2.6.2 | Device: GPU:0
Model Artifacts Dir : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\models\phase_04\paddleocr_recognition
Manifests Present   : Train=True | Val=True | Test=True


### 2. DBNet vs. SVTR Architectural Audit & Fine-Tuning Configuration


In [2]:
PADDLEOCR_TRAINING_CONFIG = {
    'model_name': 'PP-OCRv6_medium_rec',
    'architecture': 'LCNetV4 + EncoderWithLightSVTR + CTC (18,710 classes)',
    'detection_model': 'PP-OCRv6_medium_det (DBNet — Pretrained / Inference-Only)',
    'pretrained_rec_dir': str(PRETRAINED_REC_DIR),
    'pretrained_det_dir': str(PRETRAINED_DET_DIR),
    'max_epochs': 10,
    'batch_size': 8,
    'eval_batch_size': 16,
    'learning_rate': 5e-5,
    'min_learning_rate': 1e-6,
    'weight_decay': 1e-4,
    'grad_clip_norm': 5.0,
    'early_stopping_patience': 4,
    'resume_from_checkpoint': True,
    'img_height': 48,
    'img_width': 640,
    'ctc_time_steps': 80,
    'max_label_length': 75,
    'num_workers': 0,
    'seed': RANDOM_SEED,
    'device': ACTIVE_DEVICE,
}

with open(CONFIG_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(PADDLEOCR_TRAINING_CONFIG, f, sort_keys=False)

df_arch_audit = pd.DataFrame([
    {
        'Component': 'Text Detection',
        'Model': 'PP-OCRv6_medium_det',
        'Architecture': 'LCNetV4 + RepLKFPN + DBNet',
        'Checkpoint Exists': (PRETRAINED_DET_DIR / 'inference.pdiparams').exists(),
        'Status in Notebook': 'PRETRAINED / INFERENCE-ONLY',
    },
    {
        'Component': 'Text Recognition',
        'Model': 'PP-OCRv6_medium_rec',
        'Architecture': 'LCNetV4 + EncoderWithLightSVTR + CTC',
        'Checkpoint Exists': (PRETRAINED_REC_DIR / 'inference.pdiparams').exists(),
        'Status in Notebook': 'FINE-TUNING TARGET (CTC + AdamW + CosineAnnealing)',
    },
])
display(df_arch_audit)
display(pd.DataFrame([{'Parameter': k, 'Value': str(v)} for k, v in PADDLEOCR_TRAINING_CONFIG.items()]))


,Component,Model,Architecture,Checkpoint Exists,Status in Notebook
0,Text Detection,PP-OCRv6_medium_det,LCNetV4 + RepLKFPN + DBNet,True,PRETRAINED / INFERENCE-ONLY
1,Text Recognition,PP-OCRv6_medium_rec,LCNetV4 + EncoderWithLightSVTR + CTC,True,FINE-TUNING TARGET (CTC + AdamW + CosineAnneal...


,Parameter,Value
0,model_name,PP-OCRv6_medium_rec
1,architecture,"LCNetV4 + EncoderWithLightSVTR + CTC (18,710 c..."
2,detection_model,PP-OCRv6_medium_det (DBNet — Pretrained / Infe...
3,pretrained_rec_dir,C:\Users\ASUS TUF\.paddlex\official_models\PP-...
4,pretrained_det_dir,C:\Users\ASUS TUF\.paddlex\official_models\PP-...
5,max_epochs,10
6,batch_size,8
7,eval_batch_size,16
8,learning_rate,5e-05
9,min_learning_rate,1e-06


### 3. Metrics, Vocabulary, Dataset Loader & Trainable SVTR Builder


In [3]:
def clean_ocr_text(text) -> str:
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ''
    t = str(text)
    if t.strip().lower() == 'nan':
        return ''
    t = t.replace('c\u0304', 'with').replace('c̄', 'with')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return unicodedata.normalize('NFC', t)

def levenshtein_distance(seq1, seq2) -> int:
    n, m = len(seq1), len(seq2)
    if n == 0: return m
    if m == 0: return n
    if n > m:
        seq1, seq2 = seq2, seq1
        n, m = m, n
    prev_row = list(range(n + 1))
    for i in range(1, m + 1):
        curr_row = [i] + [0] * n
        s2_item = seq2[i - 1]
        for j in range(1, n + 1):
            cost = 0 if seq1[j - 1] == s2_item else 1
            curr_row[j] = min(prev_row[j] + 1, curr_row[j - 1] + 1, prev_row[j - 1] + cost)
        prev_row = curr_row
    return int(prev_row[n])

def compute_sample_ocr_metrics(reference: str, hypothesis: str) -> dict:
    r, h = clean_ocr_text(reference), clean_ocr_text(hypothesis)
    if len(r) == 0:
        m = (len(h) == 0)
        return {'cer': 0.0 if m else 1.0, 'wer': 0.0 if m else 1.0, 'char_acc_pct': 100.0 if m else 0.0,
                'norm_ed': 1.0 if m else 0.0, 'exact_match_pct': 100.0 if m else 0.0}
    c_dist = levenshtein_distance(r, h)
    r_len = len(r)
    cer = float(c_dist / r_len)
    char_acc_pct = float(max(0.0, (r_len - c_dist) / r_len) * 100.0)
    norm_ed = float(1.0 - (c_dist / max(r_len, len(h), 1)))
    rw, hw = r.split(), h.split()
    wer = float(levenshtein_distance(rw, hw) / max(1, len(rw)))
    return {'cer': cer, 'wer': wer, 'char_acc_pct': char_acc_pct, 'norm_ed': norm_ed,
            'exact_match_pct': 100.0 if r == h else 0.0}

def aggregate_ocr_metrics(refs: list, hyps: list) -> dict:
    if not refs:
        return {'num_samples': 0, 'char_acc_pct': 0.0, 'cer': 0.0, 'wer': 0.0, 'norm_ed': 0.0, 'exact_match_pct': 0.0}
    rows = [compute_sample_ocr_metrics(r, h) for r, h in zip(refs, hyps)]
    return {
        'num_samples': len(rows),
        'char_acc_pct': float(np.mean([x['char_acc_pct'] for x in rows])),
        'cer': float(np.mean([x['cer'] for x in rows])),
        'wer': float(np.mean([x['wer'] for x in rows])),
        'norm_ed': float(np.mean([x['norm_ed'] for x in rows])),
        'exact_match_pct': float(np.mean([x['exact_match_pct'] for x in rows])),
    }

def load_ppocrv6_lexicon(rec_dir: Path):
    with open(rec_dir / 'inference.yml', 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    char_dict = list(cfg['PostProcess']['character_dict'])
    full_chars = ['<blank>'] + char_dict + [' ']
    char2idx = {c: i + 1 for i, c in enumerate(char_dict)}
    char2idx[' '] = len(full_chars) - 1
    return full_chars, char2idx

FULL_CHAR_LIST, CHAR2IDX = load_ppocrv6_lexicon(PRETRAINED_REC_DIR)
NUM_CTC_CLASSES = len(FULL_CHAR_LIST)

def preprocess_crop(img: np.ndarray, target_h: int = 48, target_w: int = 640) -> np.ndarray:
    if img is None or img.size == 0:
        return np.zeros((3, target_h, target_w), dtype=np.float32)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    h, w = img.shape[:2]
    rw = min(target_w, max(1, int(math.ceil(target_h * (w / float(max(1, h)))))))
    resized = cv2.resize(img, (rw, target_h), interpolation=cv2.INTER_AREA if rw < w else cv2.INTER_LINEAR)
    norm = (resized.astype(np.float32) / 255.0 - 0.5) / 0.5
    out = np.zeros((3, target_h, target_w), dtype=np.float32)
    out[:, :, :rw] = norm.transpose(2, 0, 1)
    return out

def truncate_ctc_tokens(token_ids: list, max_t: int = 80, max_l: int = 75) -> list:
    safe, steps = [], 0
    for tok in token_ids[:max_l]:
        add = 2 if (safe and tok == safe[-1]) else 1
        if steps + add > max_t:
            break
        safe.append(tok)
        steps += add
    return safe

def ctc_greedy_decode(probs: np.ndarray) -> list:
    best = np.argmax(probs, axis=-1)
    vsize = len(FULL_CHAR_LIST)
    out = []
    for seq in best:
        keep = np.ones(len(seq), dtype=bool)
        keep[1:] = seq[1:] != seq[:-1]
        keep &= (seq != 0)
        out.append(clean_ocr_text(''.join(FULL_CHAR_LIST[int(t)] for t in seq[keep] if 0 < int(t) < vsize)))
    return out

class OFPHILRecognitionDataset(paddle.io.Dataset):
    def __init__(self, df: pd.DataFrame):
        super().__init__()
        self.samples = []
        self.missing_files = 0
        for _, row in df.iterrows():
            gt = clean_ocr_text(row.get('ground_truth_text', ''))
            if not gt:
                continue
            enc = truncate_ctc_tokens([CHAR2IDX[c] for c in gt if c in CHAR2IDX],
                                      PADDLEOCR_TRAINING_CONFIG['ctc_time_steps'],
                                      PADDLEOCR_TRAINING_CONFIG['max_label_length'])
            if not enc:
                continue
            img_p = paths.resolve_sample_path(str(row['image_path']))
            if not img_p.exists():
                self.missing_files += 1
                continue
            bbox = json.loads(row['bounding_box']) if isinstance(row['bounding_box'], str) else row['bounding_box']
            if bbox[2] <= 0 or bbox[3] <= 0:
                continue
            self.samples.append({
                'crop_id': str(row.get('crop_id', '')),
                'image_filename': str(row.get('image_filename', img_p.name)),
                'img_path': str(img_p),
                'bbox': [int(round(float(v))) for v in bbox[:4]],
                'gt': gt,
                'enc': enc,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        it = self.samples[idx]
        bx, by, bw, bh = it['bbox']
        img = cv2.imread(it['img_path'], cv2.IMREAD_COLOR)
        if img is None:
            crop = np.zeros((48, 640, 3), dtype=np.uint8)
        else:
            ih, iw = img.shape[:2]
            x1, y1 = max(0, min(bx, iw - 1)), max(0, min(by, ih - 1))
            x2, y2 = max(x1 + 1, min(bx + bw, iw)), max(y1 + 1, min(by + bh, ih))
            crop = img[y1:y2, x1:x2]
        t_img = preprocess_crop(crop, PADDLEOCR_TRAINING_CONFIG['img_height'], PADDLEOCR_TRAINING_CONFIG['img_width'])
        lbl = np.array(it['enc'], dtype=np.int32)
        return t_img, lbl, len(lbl), it['gt'], it['image_filename'], it['crop_id']

def ocr_collate_fn(batch):
    imgs, lbls, lens, gts, fns, cids = zip(*batch)
    max_l = max(lens)
    pad = np.zeros((len(batch), max_l), dtype=np.int32)
    for i, s in enumerate(lbls):
        pad[i, :len(s)] = s
    return (
        paddle.to_tensor(np.stack(imgs, axis=0), dtype='float32'),
        paddle.to_tensor(pad, dtype='int32'),
        paddle.to_tensor(np.array(lens, dtype=np.int64), dtype='int64'),
        list(gts), list(fns), list(cids),
    )

def load_trainable_ppocrv6_rec(for_training: bool = True):
    major = int(str(paddle.__version__).split('.')[0])
    if (PRETRAINED_REC_DIR / 'inference.json').exists() and major < 3:
        raise RuntimeError(
            f"PP-OCRv6_medium_rec uses Paddle PIR format requiring PaddlePaddle >= 3.0, "
            f"but active kernel runs {paddle.__version__}. Switch kernel to Python (venv312) / Thesis GPU."
        )
    m = paddle.jit.load(str(PRETRAINED_REC_DIR / 'inference'))
    m.train() if for_training else m.eval()
    return m

def to_ctc_log_probs(out: paddle.Tensor) -> paddle.Tensor:
    if isinstance(out, (tuple, list)):
        out = out[0]
    return paddle.log(paddle.clip(out, min=1e-7, max=1.0)).transpose([1, 0, 2])

def evaluate_loader(model, loader, collect_preds: bool = False):
    was_tr = model.training
    model.eval()
    ctc = paddle.nn.CTCLoss(blank=0, reduction='mean')
    tot_loss, n_samp = 0.0, 0
    gts, preds, fns, cids = [], [], [], []
    with paddle.no_grad():
        for b_img, b_lbl, b_tl, b_gt, b_fn, b_cid in loader:
            bsz = b_img.shape[0]
            out = model(b_img)
            if isinstance(out, (tuple, list)):
                out = out[0]
            lp = to_ctc_log_probs(out)
            in_l = paddle.full([bsz], lp.shape[0], dtype='int64')
            lv = float(ctc(lp, b_lbl, in_l, b_tl).item())
            if math.isfinite(lv):
                tot_loss += lv * bsz
            n_samp += bsz
            dec = ctc_greedy_decode(out.numpy())
            gts.extend(b_gt); preds.extend(dec)
            if collect_preds:
                fns.extend(b_fn); cids.extend(b_cid)
    if was_tr:
        model.train()
    res = aggregate_ocr_metrics(gts, preds)
    res['loss'] = float(tot_loss / max(1, n_samp))
    if collect_preds:
        res['records'] = [
            {'crop_id': c, 'image_filename': f, 'ground_truth': g, 'predicted_text': p, **compute_sample_ocr_metrics(g, p)}
            for c, f, g, p in zip(cids, fns, gts, preds)
        ]
    return res

# Verify strict document-level split isolation
df_train_meta = pd.read_csv(TRAIN_MANIFEST)
df_val_meta = pd.read_csv(VAL_MANIFEST)
_te_docs = set(pd.read_csv(TEST_MANIFEST, usecols=['document_id'])['document_id'])
assert set(df_train_meta['document_id']).isdisjoint(set(df_val_meta['document_id']))
assert set(df_train_meta['document_id']).isdisjoint(_te_docs)
assert set(df_val_meta['document_id']).isdisjoint(_te_docs)
print(f"[OK] Lexicon={NUM_CTC_CLASSES:,} | Zero document leakage across Train ({len(df_train_meta):,}), Val ({len(df_val_meta):,}), Test (untouched).")


[OK] Lexicon=18,710 | Zero document leakage across Train (15,043), Val (3,536), Test (untouched).


### 4. Fine-Tuning Loop with Checkpointing & Safe Resume


In [4]:
def atomic_save(obj, path: Path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    paddle.save(obj, str(tmp))
    os.replace(str(tmp), str(path))

def save_ckpt(ckpt_dir: Path, model, opt, sched, meta: dict):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    atomic_save(model.state_dict(), ckpt_dir / 'model_weights.pdparams')
    atomic_save(opt.state_dict(), ckpt_dir / 'optimizer.pdopt')
    atomic_save(sched.state_dict(), ckpt_dir / 'scheduler.pdstates')
    for fn in ('inference.yml', 'inference.json'):
        src = PRETRAINED_REC_DIR / fn
        dst = ckpt_dir / fn
        if src.exists() and not dst.exists():
            shutil.copy2(str(src), str(dst))
    with open(ckpt_dir / 'checkpoint_meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)

train_ds = OFPHILRecognitionDataset(df_train_meta)
val_ds = OFPHILRecognitionDataset(df_val_meta)
print(f"Train crops resolved: {len(train_ds):,} (missing={train_ds.missing_files}) | Val crops resolved: {len(val_ds):,} (missing={val_ds.missing_files})")

training_history = []
try:
    if len(train_ds) == 0 or len(val_ds) == 0:
        raise RuntimeError("No resolvable training/validation image crops found.")

    train_loader = paddle.io.DataLoader(train_ds, batch_size=PADDLEOCR_TRAINING_CONFIG['batch_size'], shuffle=True, collate_fn=ocr_collate_fn)
    val_loader = paddle.io.DataLoader(val_ds, batch_size=PADDLEOCR_TRAINING_CONFIG['eval_batch_size'], shuffle=False, collate_fn=ocr_collate_fn)

    rec_model = load_trainable_ppocrv6_rec(for_training=True)
    MAX_EPOCHS = PADDLEOCR_TRAINING_CONFIG['max_epochs']
    lr_sched = paddle.optimizer.lr.CosineAnnealingDecay(
        learning_rate=PADDLEOCR_TRAINING_CONFIG['learning_rate'],
        T_max=max(1, MAX_EPOCHS * len(train_loader)),
        eta_min=PADDLEOCR_TRAINING_CONFIG['min_learning_rate'],
    )
    opt = paddle.optimizer.AdamW(
        learning_rate=lr_sched,
        parameters=[p for p in rec_model.parameters() if not p.stop_gradient],
        weight_decay=PADDLEOCR_TRAINING_CONFIG['weight_decay'],
        grad_clip=paddle.nn.ClipGradByGlobalNorm(PADDLEOCR_TRAINING_CONFIG['grad_clip_norm']),
    )
    ctc_fn = paddle.nn.CTCLoss(blank=0, reduction='mean')

    start_epoch, best_cer, best_epoch, no_imp = 1, float('inf'), 0, 0
    if (BEST_CKPT_DIR / 'checkpoint_meta.json').exists():
        bm = json.loads((BEST_CKPT_DIR / 'checkpoint_meta.json').read_text(encoding='utf-8'))
        best_cer = float(bm.get('val_cer', float('inf')))
        best_epoch = int(bm.get('epoch', 0))
    if PADDLEOCR_TRAINING_CONFIG['resume_from_checkpoint'] and (LAST_CKPT_DIR / 'model_weights.pdparams').exists() and (LAST_CKPT_DIR / 'checkpoint_meta.json').exists():
        lm = json.loads((LAST_CKPT_DIR / 'checkpoint_meta.json').read_text(encoding='utf-8'))
        rec_model.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'model_weights.pdparams')))
        lr_sched.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'scheduler.pdstates')))
        opt.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'optimizer.pdopt')))
        start_epoch = int(lm.get('epoch', 0)) + 1
        no_imp = int(lm.get('epochs_without_improvement', 0))
        if HISTORY_CSV_PATH.exists():
            training_history = pd.read_csv(HISTORY_CSV_PATH).to_dict('records')
        print(f"[RESUME] Resuming from Epoch {start_epoch} (Best Epoch={best_epoch}, Best Val CER={best_cer:.4f})")

    if start_epoch <= MAX_EPOCHS:
        PIPELINE_STATUS = 'TRAINING IN PROGRESS'
        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            t0 = time.time()
            rec_model.train()
            run_loss, seen = 0.0, 0
            for step, (b_img, b_lbl, b_tl, _, _, _) in enumerate(train_loader, 1):
                bsz = b_img.shape[0]
                lp = to_ctc_log_probs(rec_model(b_img))
                in_l = paddle.full([bsz], lp.shape[0], dtype='int64')
                loss = ctc_fn(lp, b_lbl, in_l, b_tl)
                lv = float(loss.item())
                if not math.isfinite(lv):
                    opt.clear_grad()
                    continue
                loss.backward()
                opt.step()
                opt.clear_grad()
                lr_sched.step()
                run_loss += lv * bsz
                seen += bsz

            if HAS_CUDA:
                paddle.device.cuda.empty_cache()
            gc.collect()

            tr_loss = float(run_loss / max(1, seen))
            vm = evaluate_loader(rec_model, val_loader, collect_preds=False)
            dur = float(time.time() - t0)
            cur_lr = float(lr_sched.get_lr())
            vram = get_gpu_vram_mb()

            is_best = vm['cer'] < best_cer - 1e-6
            if is_best:
                best_cer, best_epoch, no_imp = vm['cer'], epoch, 0
                save_ckpt(BEST_CKPT_DIR, rec_model, opt, lr_sched, {'epoch': epoch, 'val_cer': round(best_cer, 6), 'val_char_acc_pct': round(vm['char_acc_pct'], 2)})
            else:
                no_imp += 1

            save_ckpt(LAST_CKPT_DIR, rec_model, opt, lr_sched, {'epoch': epoch, 'best_epoch': best_epoch, 'best_val_cer': round(best_cer, 6), 'epochs_without_improvement': no_imp})

            training_history.append({
                'epoch': epoch, 'train_loss': round(tr_loss, 4), 'val_loss': round(vm['loss'], 4),
                'val_char_acc_pct': round(vm['char_acc_pct'], 2), 'val_cer': round(vm['cer'], 4),
                'val_wer': round(vm['wer'], 4), 'val_norm_ed': round(vm['norm_ed'], 4),
                'val_exact_match_pct': round(vm['exact_match_pct'], 2), 'learning_rate': cur_lr,
                'duration_sec': round(dur, 2), 'gpu_vram_mb': round(vram, 1), 'is_best': is_best,
            })
            pd.DataFrame(training_history).to_csv(HISTORY_CSV_PATH, index=False)
            print(f"Epoch {epoch:02d}/{MAX_EPOCHS:02d} | TrLoss={tr_loss:.4f} | ValLoss={vm['loss']:.4f} | Acc={vm['char_acc_pct']:.2f}% | CER={vm['cer']:.4f} | WER={vm['wer']:.4f} | 1-NED={vm['norm_ed']:.4f} | Exact={vm['exact_match_pct']:.2f}% | {dur:.1f}s")
            if no_imp >= PADDLEOCR_TRAINING_CONFIG['early_stopping_patience']:
                print(f"[EARLY STOPPING] No CER improvement for {no_imp} epochs.")
                break

    PIPELINE_STATUS = 'TRAINING COMPLETED'
    with open(TRAINING_STATE_JSON, 'w', encoding='utf-8') as f:
        json.dump({'status': PIPELINE_STATUS, 'best_epoch': best_epoch, 'best_val_cer': best_cer}, f, indent=2)
except Exception as exc:
    PIPELINE_STATUS = 'FAILED – SEE ERROR'
    PIPELINE_ERROR = f"{type(exc).__name__}: {exc}"
    raise


Train crops resolved: 15,043 (missing=0) | Val crops resolved: 3,536 (missing=0)


RuntimeError: PP-OCRv6_medium_rec uses Paddle PIR format requiring PaddlePaddle >= 3.0, but active kernel runs 2.6.2. Switch kernel to Python (venv312) / Thesis GPU.

### 5. Held-Out Test Evaluation & Structured OCR Entity Export


In [ ]:
CURRENCY_PATTERN = re.compile(r'(?:PHP|Php|P|\u20B1)?\s*([0-9]{1,3}(?:,[0-9]{3})*(?:\.[0-9]{2})?)')
DATE_PATTERN = re.compile(r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b')
INDIGENCY_KEYWORD = re.compile(r'\b(indigent|indigency|katibayan|social service|financial assistance|dswd|4ps|pwd)\b', re.IGNORECASE)

def parse_entities(raw_text: str) -> dict:
    return {
        'detected_amounts': [m for m in CURRENCY_PATTERN.findall(raw_text) if m and len(m) > 1],
        'detected_dates': DATE_PATTERN.findall(raw_text),
        'has_indigency_cues': bool(INDIGENCY_KEYWORD.search(raw_text)),
    }

# 1. Held-out test evaluation strictly using BEST_CKPT_DIR
best_weights = BEST_CKPT_DIR / 'model_weights.pdparams'
if best_weights.exists() and TEST_MANIFEST.exists():
    df_test_meta = pd.read_csv(TEST_MANIFEST)
    test_ds = OFPHILRecognitionDataset(df_test_meta)
    test_loader = paddle.io.DataLoader(test_ds, batch_size=PADDLEOCR_TRAINING_CONFIG['eval_batch_size'], shuffle=False, collate_fn=ocr_collate_fn)
    eval_model = load_trainable_ppocrv6_rec(for_training=False)
    eval_model.set_state_dict(paddle.load(str(best_weights)))
    eval_model.eval()
    test_res = evaluate_loader(eval_model, test_loader, collect_preds=True)
    df_test_preds = pd.DataFrame(test_res['records'])
    df_test_preds.to_csv(TEST_PREDICTIONS_CSV, index=False, encoding='utf-8')

    report = {
        'model_name': PADDLEOCR_TRAINING_CONFIG['model_name'],
        'architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'detection_status': 'PP-OCRv6_medium_det (DBNet) — PRETRAINED / INFERENCE-ONLY',
        'num_evaluated_samples': test_res['num_samples'],
        'test_loss': round(test_res['loss'], 4),
        'character_accuracy_pct': round(test_res['char_acc_pct'], 2),
        'cer': round(test_res['cer'], 4),
        'wer': round(test_res['wer'], 4),
        'norm_ed': round(test_res['norm_ed'], 4),
        'exact_match_pct': round(test_res['exact_match_pct'], 2),
        'status': 'TEST EVALUATION COMPLETED',
    }
    with open(FINAL_REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2)
    PIPELINE_STATUS = 'TEST EVALUATION COMPLETED'
    display(pd.DataFrame([report]))
    display(df_test_preds[['image_filename', 'ground_truth', 'predicted_text', 'cer', 'wer']].head(10))

# 2. Export structured OCR entities JSON for downstream Notebooks 05 & 06
if paths.ocr_recognition_corpus.exists():
    corpus = json.loads(paths.ocr_recognition_corpus.read_text(encoding='utf-8'))
    extracted_registry = {}
    for doc_id, doc_data in corpus.items():
        lines_list = doc_data.get('extracted_lines', [])
        full_text = ' '.join(str(ln.get('text', '')) for ln in lines_list)
        confs = [float(ln.get('confidence', 0.0)) for ln in lines_list if ln.get('confidence') is not None]
        extracted_registry[doc_id] = {
            'document_id': doc_id,
            'file_name': doc_data.get('file_name', doc_id),
            'status': doc_data.get('status', 'SUCCESS'),
            'lines_count': len(lines_list),
            'average_confidence': round(float(np.mean(confs)) if confs else 0.0, 4),
            'full_text_length': len(full_text),
            'full_text_preview': full_text[:300],
            **parse_entities(full_text),
        }
    with open(OUT_ENTITIES_JSON, 'w', encoding='utf-8') as f:
        json.dump(extracted_registry, f, indent=2)
    print(f"[OK] Exported structured OCR entities ({len(extracted_registry)} docs) -> {OUT_ENTITIES_JSON}")
